# EBI BioImage Archive — add MicroManager metadata files

**Dataset:** 2026 Multicolour Bayer-SMLM paper (accession `S-BIAD3210`)
**Source:** SMB share, mounted at `/scratch/sycamore-asap/` on this machine (as for `EBI_Upload_Update.ipynb`)
**Destination:** `ftp-private.ebi.ac.uk` (BioImage Archive Webin FTP)

## Why this notebook exists

The deposition so far contains only the raw `.tif`/`.ome.tif` files. MicroManager's
`*_metadata.txt` sidecar next to each acquisition was never uploaded. The analysis code reads the
camera ROI (sensor offset + size) from that file (`IO_Functions.metadata_reader_imageJ`,
`Helper_Functions.load_metadata_roi`) to crop the per-pixel calibration maps. The paper's
acquisitions are camera-ROI crops (e.g. Ximea S. aureus `336-0-680-684` on a 2064x1544 sensor;
ZWO S. aureus `1268-480-1064-1080` on 3840x2160), so without it an external reader can't fit the
deposited data correctly — `fit_imaging_data` stops with `No metadata files found`.

## What this notebook does, in order

1. **Load the manifest** `2026_Multicolour_ebi_filelist.tsv` (ground truth for what is on EBI).
2. **Find the metadata files** — every `*metadata.txt` in each folder that has deposited TIFFs,
   and a **coverage report**: any deposited TIFF series without a matching metadata file.
   Nothing is written or uploaded in this step.
3. **Hash** the metadata files not yet in the manifest.
4. **Merge** them into the manifest (the current one is backed up as `.bak` first).
5. **Upload** them (size-based resume; safe to re-run).
6. **Verify** every metadata file is on the remote.
7. **Rebuild + re-upload** `biostudies_checksums.json`.
8. **Rebuild** `2026_Multicolour_Filelist.json` — submit it through the portal by hand.

## Credentials

Nothing is hard-coded here (this notebook is tracked in git). They are read from a local
credentials file — by default `~/.config/pyS3M/ebi_ftp_credentials.txt`, or wherever
`EBI_CREDENTIALS_FILE` points — see `_ebi_credentials.py` for the format.

---
## 0 — Imports

In [ ]:
import csv
import ftplib
import hashlib
import json
import os
import re
import shutil
import sys
import time
from collections import defaultdict
from datetime import timedelta
from pathlib import Path

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not available — progress will be printed every 10 files")

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
# ── Source data (share mounted at /scratch/ on this machine) ───────────────────
SMB_ROOT = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/Data')
OUT_DIR  = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload')

# ── Manifest = what is on EBI; extended in place (backed up first) ────────────
BASELINE_TSV = OUT_DIR / "2026_Multicolour_ebi_filelist.tsv"

# ── Session outputs ────────────────────────────────────────────────────────────
METADATA_UPLOAD_PAIRS = OUT_DIR / "pending_metadata_upload_paths.txt"
CHECKSUMS_JSON        = OUT_DIR / "biostudies_checksums.json"
FILELIST_JSON         = OUT_DIR / "2026_Multicolour_Filelist.json"

# ── EBI path prefix (preserved inside the submission) ─────────────────────────
EBI_PREFIX = "2026_Multicolour_Paper"
ACCESSION  = "S-BIAD3210"

# ── MicroManager sidecar naming: <acquisition stem>_metadata.txt ──────────────
METADATA_SUFFIX = "metadata.txt"

# ── FTP credentials (EBI-issued) ────────────────────────────────────────────
# Read from a local credentials file -- never stored in this git-tracked notebook.
# See _ebi_credentials.py for where the file lives and its format.
from _ebi_credentials import load_ebi_credentials
_creds   = load_ebi_credentials()
FTP_HOST = "ftp-private.ebi.ac.uk"
FTP_USER = _creds["WEBIN_USER"]
FTP_PASS = _creds["WEBIN_PASS"]
FTP_ROOT = _creds["EBI_FTP_DIR"]   # upload root: the submission's private FTP directory

# ── Sanity check ──────────────────────────────────────────────────────────────
for label, p in [("Data folder", SMB_ROOT), ("Manifest", BASELINE_TSV)]:
    print(f"{label:<12}: {p}  {'OK' if p.exists() else 'NOT FOUND — check the /scratch/ mount'}")
print(f"FTP host    : {FTP_HOST}")
print(f"FTP user    : {FTP_USER}")

---
## Helper functions

Hashing, size formatting and FTP plumbing are the same as in `EBI_Upload_Update.ipynb`.
`find_metadata_files` / `coverage_report` are new.

In [ ]:
def md5sum(path: Path, chunk: int = 1 << 20) -> str:
    """Return hex MD5 of a file, reading in 1 MB chunks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


def human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


def load_manifest(path: Path) -> list:
    """Read the TSV manifest into a list of {filename, md5, size} dicts."""
    rows = []
    with path.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        name_key = "filename" if "filename" in (reader.fieldnames or []) else "files"
        for row in reader:
            rows.append({"filename": row[name_key], "md5": row["md5"], "size": int(row["size"])})
    return rows


def write_manifest(rows: list, out_path: Path) -> None:
    """Write {filename, md5, size} rows to a TSV manifest, sorted by filename."""
    with out_path.open("w") as f:
        f.write("filename\tmd5\tsize\n")
        for r in sorted(rows, key=lambda r: r["filename"]):
            f.write(f"{r['filename']}\t{r['md5']}\t{r['size']}\n")


TIFF_RE = re.compile(r"\.(ome\.)?tiff?$", re.IGNORECASE)


def is_covered(tiff_name: str, meta_stems: set) -> bool:
    """True if a '<stem>_metadata.txt' belongs to this TIFF: the stem is the TIFF name without
    its extension, or that name minus a trailing _1, _2 ... continuation-file suffix. Grid
    positions ('..._MMStack_2-Pos000_001') keep their own stem."""
    name = TIFF_RE.sub("", tiff_name)
    base = re.sub(r"_\d+$", "", name)
    return name in meta_stems or base in meta_stems


def find_metadata_files(smb_root: Path, ebi_prefix: str, manifest_rows: list) -> dict:
    """For every folder holding deposited TIFFs, list the *metadata.txt files beside them.

    Returns {folder (relative to smb_root): {"tiffs": [names], "metadata": [Paths]}}.
    """
    folders = defaultdict(lambda: {"tiffs": [], "metadata": []})
    for r in manifest_rows:
        rel = Path(r["filename"]).relative_to(ebi_prefix)
        if TIFF_RE.search(rel.name):
            folders[rel.parent]["tiffs"].append(rel.name)
    for rel_folder, entry in folders.items():
        local = smb_root / rel_folder
        if local.is_dir():
            entry["metadata"] = sorted(
                p for p in local.iterdir() if p.is_file() and p.name.endswith(METADATA_SUFFIX)
            )
    return dict(folders)


def coverage_report(folders: dict) -> list:
    """Deposited TIFFs with no matching *metadata.txt beside them."""
    uncovered = []
    for rel_folder, entry in sorted(folders.items()):
        meta_stems = {m.name[: -len(METADATA_SUFFIX)].rstrip("_") for m in entry["metadata"]}
        uncovered += [f"{rel_folder}/{t}" for t in sorted(entry["tiffs"]) if not is_covered(t, meta_stems)]
    return uncovered


def remote_size(ftp: ftplib.FTP, path: str) -> int:
    """Return remote file size, or -1 if the file does not exist."""
    try:
        return ftp.size(path)
    except ftplib.error_perm:
        return -1


def ensure_remote_dirs(ftp: ftplib.FTP, remote_path: str):
    """Recursively create all intermediate directories on the FTP server."""
    parts = Path(remote_path).parent.parts
    for i in range(1, len(parts) + 1):
        try:
            ftp.mkd(str(Path(*parts[:i])))
        except ftplib.error_perm:
            pass  # already exists — fine


def ftp_connect() -> ftplib.FTP:
    ftp = ftplib.FTP(FTP_HOST, timeout=60)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)
    return ftp


def upload_file(ftp: ftplib.FTP, local: Path, remote: str, chunk: int = 1 << 20) -> str:
    """Upload local → remote, skipping if sizes match. Returns 'skip'/'ok'."""
    if remote_size(ftp, remote) == local.stat().st_size:
        return "skip"
    ensure_remote_dirs(ftp, remote)
    with local.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=chunk)
    return "ok"


def run_upload(pairs: list) -> dict:
    """Upload (local Path, manifest-relative remote path) pairs, reconnecting after errors."""
    n = len(pairs)
    print(f"Uploading {n} files to {FTP_HOST} …")
    ftp = ftp_connect()
    counts = {"ok": 0, "skip": 0, "error": 0}
    t0 = time.time()
    iterator = tqdm(enumerate(pairs, 1), total=n, unit="file") if HAS_TQDM else enumerate(pairs, 1)
    for i, (local, rel_remote) in iterator:
        try:
            status = upload_file(ftp, local, f"{FTP_ROOT}/{rel_remote}")
        except Exception as exc:
            status = "error"
            print(f"  ERROR [{i}/{n}] {local.name}: {exc}", file=sys.stderr)
            try:
                ftp.quit()
            except Exception:
                pass
            time.sleep(5)
            ftp = ftp_connect()
        counts[status] += 1
        if not HAS_TQDM and i % 10 == 0:
            print(f"  [{i}/{n}]  ok={counts['ok']}  skip={counts['skip']}  err={counts['error']}")
    try:
        ftp.quit()
    except Exception:
        pass
    print(f"\nFinished in {timedelta(seconds=int(time.time() - t0))}: "
          f"uploaded {counts['ok']}, skipped {counts['skip']}, errors {counts['error']}")
    return counts

---
## 1 — Load the manifest (what is on EBI)

Read-only.

In [ ]:
manifest_rows = load_manifest(BASELINE_TSV)
manifest_paths = {r["filename"] for r in manifest_rows}
print(f"Manifest files : {len(manifest_rows):,} ({human_size(sum(r['size'] for r in manifest_rows))})")
already_meta = sorted(p for p in manifest_paths if p.endswith(METADATA_SUFFIX))
print(f"Metadata files already in the manifest : {len(already_meta)}")

---
## 2 — Find the metadata files (dry run — nothing written or uploaded)

Lists every `*metadata.txt` next to deposited TIFFs. **Check the coverage report before going on:**
a TIFF listed there has no matching metadata file on the share, so a reader still won't be
able to get its ROI. Fix those by hand (find the sidecar, or note the series as unfittable).

In [ ]:
folders = find_metadata_files(SMB_ROOT, EBI_PREFIX, manifest_rows)

missing_folders = [f for f in folders if not (SMB_ROOT / f).is_dir()]
if missing_folders:
    print(f"WARNING: {len(missing_folders)} manifest folder(s) not found under {SMB_ROOT}:")
    for f in missing_folders[:20]:
        print(" ", f)

meta_files = [m for e in folders.values() for m in e["metadata"]]
print(f"Folders with deposited TIFFs : {len(folders):,}")
print(f"Metadata files found         : {len(meta_files):,} ({human_size(sum(m.stat().st_size for m in meta_files))})")
print()
print(f"{'Folder':<90} {'TIFFs':>6} {'metadata':>9}")
for rel_folder, e in sorted(folders.items()):
    print(f"{str(rel_folder):<90} {len(e['tiffs']):>6} {len(e['metadata']):>9}")

uncovered = coverage_report(folders)
print()
if uncovered:
    print(f"COVERAGE: {len(uncovered)} deposited TIFF(s) with no matching *_metadata.txt:")
    for u in uncovered:
        print(" ", u)
else:
    print("COVERAGE: every deposited TIFF has a matching metadata file.")

---
## 3 — Hash the metadata files not yet in the manifest

Some of these sidecars are large (GB-scale for long acquisitions), so hashing over SMB takes a while.

In [ ]:
new_rows, pairs = [], []
to_hash = [m for m in meta_files if f"{EBI_PREFIX}/{m.relative_to(SMB_ROOT)}" not in manifest_paths]
print(f"{len(to_hash)} metadata file(s) to hash ({len(meta_files) - len(to_hash)} already in the manifest)")

t0 = time.time()
iterator = tqdm(to_hash, unit="file") if HAS_TQDM else to_hash
for src in iterator:
    ebi_path = f"{EBI_PREFIX}/{src.relative_to(SMB_ROOT)}"
    new_rows.append({"filename": ebi_path, "md5": md5sum(src), "size": src.stat().st_size})
    pairs.append((src, ebi_path))

with METADATA_UPLOAD_PAIRS.open("w") as pp:
    for src, ebi_path in pairs:
        pp.write(f"{src}\t{ebi_path}\n")
print(f"Done in {timedelta(seconds=int(time.time() - t0))}. Upload pairs: {METADATA_UPLOAD_PAIRS}")

---
## 4 — Merge into the manifest

Backs up `2026_Multicolour_ebi_filelist.tsv` as `.bak`, then rewrites it with the metadata rows added.

In [ ]:
if new_rows:
    backup_path = BASELINE_TSV.with_suffix(BASELINE_TSV.suffix + ".bak")
    # copyfile, not copy/copy2: this SMB mount rejects the chmod/utime those try afterwards.
    shutil.copyfile(BASELINE_TSV, backup_path)
    print(f"Backed up manifest to {backup_path}")
    merged_rows = manifest_rows + new_rows
    write_manifest(merged_rows, BASELINE_TSV)
    print(f"Merged manifest : {len(merged_rows):,} files ({human_size(sum(r['size'] for r in merged_rows))})")
else:
    merged_rows = manifest_rows
    print("No new rows — manifest left unchanged.")

---
## 5 — Upload the metadata files (FTP)

Size-based resume: re-running skips anything already uploaded at the correct size. Uploads every
metadata file found in §2, not only the newly hashed ones, so a re-run also finishes any upload
interrupted in an earlier session.

In [ ]:
all_pairs = [(m, f"{EBI_PREFIX}/{m.relative_to(SMB_ROOT)}") for m in meta_files]
counts = run_upload(all_pairs)

---
## 6 — Verify every metadata file is on the remote, at the right size

In [ ]:
ftp = ftp_connect()
wrong = []
for local, rel_remote in all_pairs:
    if remote_size(ftp, f"{FTP_ROOT}/{rel_remote}") != local.stat().st_size:
        wrong.append(rel_remote)
ftp.quit()
if wrong:
    print(f"MISMATCH — {len(wrong)} metadata file(s) missing or the wrong size on the remote:")
    for w in wrong[:20]:
        print(" ", w)
    print("Re-run §5.")
else:
    print(f"MATCH — all {len(all_pairs)} metadata files present on the remote at the correct size.")

---
## 7 — Rebuild the checksums manifest and re-upload it

Same format as `EBI_Upload_Update.ipynb` §7, now covering the metadata files too.

In [ ]:
def build_checksums_json(rows: list, out_path: Path, accession: str) -> dict:
    """Build biostudies_checksums.json ({path: {md5, size}}) from manifest rows."""
    files = {r["filename"]: {"md5": r["md5"], "size": r["size"]} for r in rows}
    manifest = {
        "_meta": {
            "description": (
                "MD5 checksums for the raw acquisition files (and their MicroManager "
                f"metadata.txt sidecars) deposited to BioStudies {accession} (2026 Multicolour "
                "Bayer-SMLM paper) -- a curated subset of the full acquired dataset. Computed "
                "from the original local files before upload "
                "(notebooks/EBI_Upload/EBI_Upload_Notebook.ipynb, EBI_Upload_Update.ipynb, "
                "EBI_Upload_Metadata.ipynb), so a mismatch after download indicates real "
                "corruption/truncation."
            ),
            "accession": accession,
            "algorithm": "md5",
            "n_files": len(files),
            "total_bytes": sum(r["size"] for r in rows),
        },
        "files": files,
    }
    out_path.write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {out_path}  ({len(files):,} files)")
    return manifest


build_checksums_json(merged_rows, CHECKSUMS_JSON, ACCESSION)

remote = f"{FTP_ROOT}/{EBI_PREFIX}/Checksums/{CHECKSUMS_JSON.name}"
ftp = ftp_connect()
ensure_remote_dirs(ftp, remote)
with CHECKSUMS_JSON.open("rb") as f:
    ftp.storbinary(f"STOR {remote}", f, blocksize=1 << 20)
ok = remote_size(ftp, remote) == CHECKSUMS_JSON.stat().st_size
ftp.quit()
print("MATCH — checksums manifest uploaded." if ok else "MISMATCH — re-run this cell.")

---
## 8 — Rebuild the EBI file-list JSON (for the portal)

**Not** uploaded automatically — submit `2026_Multicolour_Filelist.json` through the BioImage
Archive portal by hand, or the new files won't appear in the public study.

In [ ]:
def build_ebi_filelist_json(rows: list, checksums_path: Path, ebi_prefix: str, out_path: Path) -> list:
    """Flat BioStudies file-listing JSON from manifest rows, plus the checksums manifest."""
    entries = [
        {"path": r["filename"], "size": r["size"],
         "attributes": [{"name": "size", "value": str(r["size"])}], "type": "file"}
        for r in sorted(rows, key=lambda r: r["filename"])
    ]
    cksum_size = checksums_path.stat().st_size
    entries.append({
        "path": f"{ebi_prefix}/Checksums/{checksums_path.name}", "size": cksum_size,
        "attributes": [{"name": "size", "value": str(cksum_size)}], "type": "file",
    })
    out_path.write_text(json.dumps(entries, indent=2))
    print(f"Wrote {out_path}  ({len(entries):,} entries)")
    return entries


build_ebi_filelist_json(merged_rows, CHECKSUMS_JSON, EBI_PREFIX, FILELIST_JSON)

---
## Checklist

- [ ] Running on a machine with the share mounted at `/scratch/sycamore-asap/...`
- [ ] Credentials file copied to this machine (`~/.config/pyS3M/ebi_ftp_credentials.txt`, `chmod 600`)
- [ ] §2 coverage report reviewed — every deposited TIFF series has a metadata file, or the gaps are understood
- [ ] Metadata files hashed (§3) and merged into the manifest, `.bak` written (§4)
- [ ] Uploaded (§5), errors = 0
- [ ] Verified on the remote at the correct size (§6)
- [ ] `biostudies_checksums.json` rebuilt and re-uploaded (§7)
- [ ] `2026_Multicolour_Filelist.json` rebuilt (§8) **and submitted through the BioImage Archive portal**
- [ ] Spot-check a download: `https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/210/S-BIAD3210/Files/2026_Multicolour_Paper/<folder>/<stem>_metadata.txt`